# 개별종목 조합E — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합E 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합E의 피처 값만 지정합니다.
import json

COMBINATION = 'E'
FEATURE_COLUMNS = (
    'sector_ret_5',
    'sector_ret_20',
    'relative_ret_5_sector',
    'relative_ret_20_sector',
    'relative_ret_5_market',
    'sector_hv_20',
    'sector_beta_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 24개 노트북이 각각 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20100331 ~ 20240822
학습 행·종목: 172058 162
조합E 피처: ('sector_ret_5', 'sector_ret_20', 'relative_ret_5_sector', 'relative_ret_20_sector', 'relative_ret_5_market', 'sector_hv_20', 'sector_beta_60')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130410,20130705,0.3889,0.3701,0.0187,0.3694,0.2401,0.3177
1,2,balanced,999,20140414,20140711,0.4396,0.4743,-0.0347,0.3112,0.1235,0.2208
2,3,balanced,1248,20150421,20150716,0.3558,0.3301,0.0257,0.3527,0.3253,0.3440
3,4,balanced,1496,20160422,20160719,0.3838,0.4107,-0.0269,0.3461,0.1986,0.2849
4,5,balanced,1745,20170424,20170721,0.3763,0.4177,-0.0414,0.3136,0.2085,0.2819
5,6,balanced,1994,20180503,20180731,0.4003,0.3908,0.0095,0.3951,0.3581,0.3836
6,7,balanced,2243,20190514,20190806,0.3915,0.4612,-0.0697,0.3269,0.1378,0.2331
7,8,balanced,2492,20200518,20200807,0.3668,0.3144,0.0525,0.3587,0.4144,0.3784
8,9,balanced,2741,20210518,20210810,0.3948,0.4418,-0.0470,0.3568,0.2638,0.3287
9,10,balanced,2989,20220519,20220812,0.3417,0.3343,0.0073,0.3332,0.2363,0.2953


,OOS 폴드 평균
accuracy,0.3835
training_majority_baseline_accuracy,0.3846
accuracy_minus_training_majority_baseline,-0.0010
macro_f1,0.3508
down_recall,0.2605
core_harmonic_mean,0.3143


재실행 명령: python scripts/run_stock_model_experiment.py
